# Stock Screener

In [1]:
import pathlib

_ROOT = pathlib.Path.cwd().parent

In [2]:
import duckdb
import ipywidgets as w
import pandas as pd
from dotenv import load_dotenv
from IPython.display import display
from typing_extensions import Literal

import irp.ticker_sets as ts
from irp.features.metrics import compute_metrics

load_dotenv()
con = duckdb.connect(str(_ROOT / "data/irp.duckdb"))

fundamentals = {
    t: con.execute(f"SELECT * FROM {t}").df() for t in ("income", "balance", "cashflow")
}
# Restrict prices to tickers that have fundamentals (screener needs both)
prices = con.execute(
    'SELECT ticker, date, open, high, low, close, volume FROM prices '
    'WHERE ticker IN (SELECT DISTINCT "Ticker" FROM income)'
).df()

ticker_info = con.execute("""
    SELECT c.Ticker AS ticker, i.Sector AS sector, i.Industry AS industry
    FROM companies c
    LEFT JOIN industries i ON c.IndustryId = i.IndustryId
""").df()

metrics_cache = {}


def get_metrics(period: Literal["annual", "ttm"]) -> pd.DataFrame:
    if period not in metrics_cache:
        metrics_cache[period] = compute_metrics(
            fundamentals, prices, period, latest=True
        )
    return metrics_cache[period]


_warm = get_metrics("annual")
print(f"Loaded {_warm.shape[0]} tickers, {_warm.shape[1]} metric columns")


ts.init(con)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Loaded 4420 tickers, 48 metric columns


In [3]:
from irp.features.screener_defs import (
    DEFAULT_COLS, METRIC_DEFS, METRIC_FMT, METRIC_NAMES, TABLE_CSS, fmt_cell,
)

In [4]:
import irp.features.screener as scr
from ipyevents import Event
import webbrowser

# --- top controls ---
period_dd = w.Dropdown(
    options=[("Annual", "annual"), ("TTM", "ttm")],
    value="annual", description="Period:",
    style={"description_width": "initial"},
)
cols_sel = w.SelectMultiple(
    options=METRIC_NAMES, value=tuple(DEFAULT_COLS),
    description="Show columns:", rows=12,
    layout=w.Layout(width="320px"),
    style={"description_width": "initial"},
)
sort_dd = w.Dropdown(
    options=METRIC_NAMES, value="P/E",
    description="Sort by:", style={"description_width": "initial"},
)
sort_asc = w.Checkbox(value=True, description="Ascending")
limit_in = w.IntText(
    value=50000, description="Limit:",
    layout=w.Layout(width="140px"), style={"description_width": "initial"},
)

# --- sector / industry ---
_all_sectors   = ["(All)"] + sorted(ticker_info["sector"].dropna().unique().tolist())
_all_industries = ["(All)"] + sorted(ticker_info["industry"].dropna().unique().tolist())
sector_sel = w.SelectMultiple(
    options=_all_sectors, value=["(All)"], description="Sector:",
    rows=8, layout=w.Layout(width="260px"), style={"description_width": "initial"},
)
industry_sel = w.SelectMultiple(
    options=_all_industries, value=["(All)"], description="Industry:",
    rows=8, layout=w.Layout(width="300px"), style={"description_width": "initial"},
)

# --- watchlist ---
_watchlist: list[str] = []
watchlist_sel = w.SelectMultiple(
    options=[], description="Watchlist:", rows=8,
    layout=w.Layout(width="260px"), style={"description_width": "initial"},
)
rm_wl_btn = w.Button(description="✕ Remove", button_style="warning", layout=w.Layout(width="90px"))
yf_btn    = w.Button(description="Open Yahoo Finance", button_style="info", layout=w.Layout(width="150px"))

def _add_to_watchlist(ticker):
    if ticker and ticker not in _watchlist:
        _watchlist.append(ticker)
        watchlist_sel.options = tuple(_watchlist)

def _rm_from_watchlist(_=None):
    for t in list(watchlist_sel.value):
        _watchlist.remove(t)
    watchlist_sel.options = tuple(_watchlist)

def _open_yahoo_finance(_=None):
    for ticker in (list(watchlist_sel.value) or _watchlist):
        webbrowser.open(f"https://finance.yahoo.com/quote/{ticker}/")

rm_wl_btn.on_click(_rm_from_watchlist)
yf_btn.on_click(_open_yahoo_finance)

In [8]:
# --- table + click handler ---
table_widget = w.HTML(value="<em style='color:#888;font-size:12px'>Press Apply to see results.</em>")
_click_listener = Event(source=table_widget, watched_events=["click"])

def _on_table_click(event):
    target = event.get("target", {})
    if "scr-ticker" in target.get("className", ""):
        elem_id = target.get("id", "")
        if elem_id.startswith("ticker-"):
            _add_to_watchlist(elem_id[len("ticker-"):])

_click_listener.on_dom_event(_on_table_click)

# --- metric filter rows ---
filter_rows: list[dict] = []
filters_box = w.VBox([])

def _add_filter(_=None, metric=None, vmin=None, vmax=None):
    metric_dd = w.Dropdown(options=METRIC_NAMES, value=metric or METRIC_NAMES[0], layout=w.Layout(width="240px"))
    min_in = w.FloatText(value=vmin, description="min", layout=w.Layout(width="160px"), style={"description_width": "initial"})
    max_in = w.FloatText(value=vmax, description="max", layout=w.Layout(width="160px"), style={"description_width": "initial"})
    rm_btn = w.Button(description="x", layout=w.Layout(width="32px"), button_style="warning")
    entry  = {"box": w.HBox([metric_dd, min_in, max_in, rm_btn]), "metric": metric_dd, "min": min_in, "max": max_in}
    filter_rows.append(entry)
    rm_btn.on_click(lambda _b: (_remove_filter(entry)))
    filters_box.children = tuple(e["box"] for e in filter_rows)

def _remove_filter(entry):
    filter_rows.remove(entry)
    filters_box.children = tuple(e["box"] for e in filter_rows)

add_btn   = w.Button(description="+ Add filter", button_style="info")
apply_btn = w.Button(description="Apply", button_style="primary")
add_btn.on_click(_add_filter)

def _apply(_=None):
    try:
        df = get_metrics(period_dd.value)
        metric_filters = [(e["metric"].value, e["min"].value, e["max"].value) for e in filter_rows]
        df_view, n_matched = scr.apply_filters(
            df, ticker_info,
            list(sector_sel.value), list(industry_sel.value),
            metric_filters, sort_dd.value, sort_asc.value, limit_in.value,
        )
        table_widget.value = scr.build_table_html(df_view, list(cols_sel.value), len(df), n_matched)
    except Exception as e:
        table_widget.value = f'<pre style="color:#f88">Error: {e}</pre>'

apply_btn.on_click(_apply)
_add_filter(metric="P/E", vmin=0, vmax=20.0)
_add_filter(metric="ROE", vmin=0.15)

# --- ticker sets ---
set_name_in    = w.Text(placeholder="Set name", layout=w.Layout(width="220px"))
set_mode_radio = w.RadioButtons(options=["Replace", "Add to"], value="Replace", layout=w.Layout(width="100px"))
save_set_btn   = w.Button(description="Save as Set", button_style="success", layout=w.Layout(width="110px"))
set_status_lbl = w.Label("")
sets_dd        = w.Dropdown(
    options=[f"{name} ({n})" for name, n in ts.list_sets(con)] or ["(none)"],
    description="Existing sets:", layout=w.Layout(width="300px"), style={"description_width": "initial"},
)
load_set_btn   = w.Button(description="Load to Watchlist", button_style="info", layout=w.Layout(width="140px"))
delete_set_btn = w.Button(description="Delete", button_style="danger", layout=w.Layout(width="80px"))

def _refresh_sets():
    opts = [f"{name} ({n})" for name, n in ts.list_sets(con)]
    sets_dd.options = opts or ["(none)"]

def _save_set(_=None):
    name = set_name_in.value.strip()
    if not name:         set_status_lbl.value = "Enter a set name first."; return
    if not _watchlist:   set_status_lbl.value = "Add tickers to watchlist first."; return
    ts.save_set(name, _watchlist, con, replace=(set_mode_radio.value == "Replace"))
    _refresh_sets()
    set_status_lbl.value = f"Saved {len(_watchlist)} tickers to '{name}'."

def _load_set(_=None):
    if not sets_dd.value or sets_dd.value == "(none)": return
    set_name = sets_dd.value.split(" (")[0]
    new = [t for t in ts.get_set(set_name, con) if t not in _watchlist]
    _watchlist.extend(new)
    watchlist_sel.options = tuple(_watchlist)
    set_status_lbl.value = f"Loaded {len(new)} new tickers from '{set_name}'."

def _delete_set(_=None):
    if not sets_dd.value or sets_dd.value == "(none)": return
    set_name = sets_dd.value.split(" (")[0]
    ts.delete_set(set_name, con)
    _refresh_sets()
    set_status_lbl.value = f"Deleted set '{set_name}'."

save_set_btn.on_click(_save_set)
load_set_btn.on_click(_load_set)
delete_set_btn.on_click(_delete_set)

display(
    w.HBox([w.VBox([period_dd, sort_dd, sort_asc, limit_in]), cols_sel]),
    w.HBox([
        w.VBox([w.Label("Sector:"), sector_sel]),
        w.VBox([w.Label("Industry:"), industry_sel]),
        w.VBox([w.Label("Watchlist:"), watchlist_sel, w.HBox([rm_wl_btn, yf_btn])]),
    ]),
    w.VBox([
        w.Label("Save watchlist as Ticker Set:"),
        w.HBox([set_name_in, set_mode_radio, save_set_btn]),
        w.HBox([sets_dd, load_set_btn, delete_set_btn]),
        set_status_lbl,
    ]),
    w.Label("Metric filters (AND-joined):"),
    filters_box,
    w.HBox([add_btn, apply_btn]),
    table_widget,
)

Label(value='Metric filters (AND-joined):')

HTML(value="<em style='color:#888;font-size:12px'>Press Apply to see results.</em>")

In [ ]:
_watchlist

[]

: 